# Step 1: ACDC Cardiac Cine MRI Dataset Exploration

**Project**: Motion-Guided Self-Supervised Learning for Label-Efficient Cardiac Cine MRI Analysis

### Objectives for Step 1:
1. **Dataset Verification**: Check for presence at `data/raw/ACDC/` and guide official retrieval if missing.
2. **Dataset Inspection**: Identify patient IDs, file formats (NIfTI-1 `.nii.gz`), temporal frames, ED/ES phases, spatial dimensions, voxel spacing, and patient metadata.
3. **Temporal Analysis**: Distinguish labeled frames (ED and ES only) from unlabeled frames (all intermediate temporal cine phases) to formalize the self-supervised learning setup.
4. **Label Verification & Remapping**:
   - Original ACDC: `0=Background`, `1=RV cavity`, `2=Myocardium`, `3=LV cavity`
   - Project Specification: `0=Background`, `1=LV cavity`, `2=Myocardium`, `3=RV cavity`
5. **Quality & Integrity Checks**: Identify missing files, corrupted headers, NaNs/Infs.
6. **Artifact Generation**:
   - Machine-readable dataset inventory: `data/processed/dataset_inventory.csv`
   - Exploratory figures saved under: `results/data_exploration/`

> **Important Policy**: No model training or slice-level random splitting is performed at this stage. Temporal and patient-level integrity are strictly preserved.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in Python path
PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

from src.dataset_inspector import (
    ACDCInspector,
    get_expected_acdc_structure,
    remap_labels_to_spec,
    ACDC_TO_SPEC_LABEL_MAP,
    SPEC_LABEL_NAMES,
    ACDC_ORIGINAL_LABEL_NAMES,
    PATHOLOGY_MAP
)

# Set plotting aesthetics
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

# Configured directories
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ACDC'
RESULTS_EXPLORATION_DIR = PROJECT_ROOT / 'results' / 'data_exploration'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

RESULTS_EXPLORATION_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {DATA_RAW_DIR}")
print(f"Exploration figures path: {RESULTS_EXPLORATION_DIR}")
print(f"Processed outputs path: {PROCESSED_DIR}")

## 1. Dataset Availability & Structure Verification
We verify whether the dataset is installed at `data/raw/ACDC/`.

In [ ]:
inspector = ACDCInspector(raw_dir=str(DATA_RAW_DIR))
is_present, status_msg = inspector.verify_presence()

print("=" * 70)
print(f"DATASET PRESENCE VERIFICATION: {'AVAILABLE' if is_present else 'NOT FOUND'}")
print("=" * 70)
print(status_msg)

if not is_present:
    print("\n" + "!" * 70)
    print("DATASET ACTION REQUIRED:")
    print("Please download the ACDC challenge dataset package ('training.zip')")
    print("from the official ACDC MICCAI Challenge portal (Human Heart Project / CREATIS)")
    print("and extract it into: data/raw/ACDC/")
    print("!" * 70)
    print(get_expected_acdc_structure())

## 2. Complete Dataset Inspection & Machine-Readable Inventory
When present, we inspect all patients, parsing `Info.cfg`, measuring volume shapes, voxel spacing, and identifying labeled vs. unlabeled frames.

In [ ]:
inventory_csv_path = PROCESSED_DIR / 'dataset_inventory.csv'

if is_present:
    inventory_df = inspector.save_inventory(output_path=str(inventory_csv_path), subset='training')
    print(f"Inventory preview (first 5 rows):")
    display(inventory_df.head(5))
else:
    print("Skipping inventory generation until dataset files are extracted.")
    inventory_df = None

## 3. Dataset-Wide Statistics & Summaries
- Total patients and study counts
- Pathology group distribution
- Spatial resolutions and in-plane voxel spacing
- Slice counts and temporal frame counts

In [ ]:
if inventory_df is not None and not inventory_df.empty:
    print("=" * 60)
    print("DATASET SUMMARY STATISTICS")
    print("=" * 60)
    print(f"Total number of patients: {len(inventory_df)}")
    print(f"Total 4D cine volumes: {inventory_df['has_4d_cine'].sum()}")
    print(f"Total temporal frames available: {inventory_df['num_temporal_frames'].sum()}")
    print(f"Total labeled frames (ED+ES): {inventory_df['num_labeled_frames'].sum()}")
    print(f"Total unlabeled temporal frames: {inventory_df['num_unlabeled_frames'].sum()}")
    
    print("\nPathology Distribution:")
    for grp, count in inventory_df['pathology_group'].value_counts().items():
        full_name = PATHOLOGY_MAP.get(grp, grp)
        print(f"  {grp:6s} ({full_name}): {count} patients")
    
    print("\nSpatial Dimensions & Spacing:")
    print(f"  In-plane matrix X: [{inventory_df['volume_shape_x'].min()}, {inventory_df['volume_shape_x'].max()}] (mean: {inventory_df['volume_shape_x'].mean():.1f})")
    print(f"  In-plane matrix Y: [{inventory_df['volume_shape_y'].min()}, {inventory_df['volume_shape_y'].max()}] (mean: {inventory_df['volume_shape_y'].mean():.1f})")
    print(f"  Number of slices Z: [{inventory_df['volume_shape_z'].min()}, {inventory_df['volume_shape_z'].max()}] (mean: {inventory_df['volume_shape_z'].mean():.1f})")
    print(f"  Voxel spacing X: [{inventory_df['spacing_x'].min():.2f}, {inventory_df['spacing_x'].max():.2f}] mm")
    print(f"  Voxel spacing Y: [{inventory_df['spacing_y'].min():.2f}, {inventory_df['spacing_y'].max():.2f}] mm")
    print(f"  Slice thickness Z: [{inventory_df['spacing_z'].min():.2f}, {inventory_df['spacing_z'].max():.2f}] mm")
else:
    print("Summary statistics will be displayed once ACDC files are present.")

## 4. Visualizations & Figure Artifacts
Visualizations saved to `results/data_exploration/`:
1. `pathology_distribution.png`
2. `spatial_resolution_distribution.png`
3. `cine_motion_examples.png`
4. `ed_es_segmentations.png`

In [ ]:
if inventory_df is not None and not inventory_df.empty:
    # 1. Pathology Distribution Plot
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    group_counts = inventory_df['pathology_group'].value_counts().sort_index()
    group_labels = [f"{g}\n({PATHOLOGY_MAP.get(g, g)})" for g in group_counts.index]
    colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']
    
    bars = ax.bar(group_labels, group_counts.values, color=colors[:len(group_counts)], edgecolor='black')
    for bar, count in zip(bars, group_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(count), ha='center', va='bottom', fontweight='bold')
    ax.set_ylabel('Number of Patients')
    ax.set_title('ACDC Pathology Group Distribution', fontsize=13, fontweight='bold')
    plt.tight_layout()
    pathology_fig_path = RESULTS_EXPLORATION_DIR / 'pathology_distribution.png'
    plt.savefig(pathology_fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {pathology_fig_path}")
    
    # 2. Spatial Resolution and Spacing Histograms
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(inventory_df['spacing_x'].dropna(), bins=15, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Pixel Spacing X (mm)')
    axes[0].set_title('In-plane Resolution (X)')
    
    axes[1].hist(inventory_df['volume_shape_z'].dropna(), bins=12, color='coral', edgecolor='black')
    axes[1].set_xlabel('Number of Slices (Z)')
    axes[1].set_title('Slices per 3D Volume')
    
    axes[2].hist(inventory_df['num_temporal_frames'].dropna(), bins=12, color='mediumpurple', edgecolor='black')
    axes[2].set_xlabel('Temporal Frames (T)')
    axes[2].set_title('Temporal Frames per Patient')
    plt.tight_layout()
    spacing_fig_path = RESULTS_EXPLORATION_DIR / 'spatial_resolution_distribution.png'
    plt.savefig(spacing_fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {spacing_fig_path}")

## 5. Label Encoding & Remapping Sanity Checks
We verify that the target classes conform to:
- Label 0: Background
- Label 1: LV cavity (Left Ventricle)
- Label 2: Myocardium
- Label 3: RV cavity (Right Ventricle)

Original ACDC labels: `0=BG, 1=RV, 2=Myocardium, 3=LV`. Remapping maps raw 1 $\to$ 3 and raw 3 $\to$ 1.

In [ ]:
print("LABEL ENCODING MAPPING SPECIFICATION:")
print("-" * 50)
print("Original ACDC Convention:")
for k, v in ACDC_ORIGINAL_LABEL_NAMES.items():
    print(f"  Label {k}: {v}")

print("\nProject Target Specification:")
for k, v in SPEC_LABEL_NAMES.items():
    print(f"  Label {k}: {v}")

print(f"\nRemapping lookup: {ACDC_TO_SPEC_LABEL_MAP}")

# Verification test
dummy_raw = np.array([0, 1, 2, 3], dtype=np.int64)
dummy_remap = remap_labels_to_spec(dummy_raw)
assert dummy_remap[0] == 0, "BG mismatch"
assert dummy_remap[1] == 3, "RV mismatch (orig 1 should map to target 3)"
assert dummy_remap[2] == 2, "Myocardium mismatch (orig 2 should remain 2)"
assert dummy_remap[3] == 1, "LV mismatch (orig 3 should map to target 1)"
print("\n✓ Label remapping sanity check passed successfully.")

## 6. Verification Summary & Preprocessing Readiness
Final summary of findings and check for Step 2 readiness.

In [ ]:
print("=" * 70)
print("STEP 1: DATA EXPLORATION SANITY REPORT")
print("=" * 70)
if is_present and inventory_df is not None:
    corrupted_count = inventory_df['is_corrupted'].sum()
    missing_gt = len(inventory_df[inventory_df['num_labeled_frames'] < 2])
    print(f"Patients inspected: {len(inventory_df)}")
    print(f"Corrupted files: {corrupted_count}")
    print(f"Patients with incomplete ED/ES masks: {missing_gt}")
    if corrupted_count == 0 and missing_gt == 0:
        print("\n✓ ALL SANITY CHECKS PASSED. Ready for Step 2 (Preprocessing).")
    else:
        print("\n⚠ Issues detected. Please resolve prior to Step 2.")
else:
    print("STATUS: PENDING DATASET DOWNLOAD")
    print("The dataset verification routine has identified that raw files are not yet")
    print("placed in data/raw/ACDC/. Once files are downloaded, rerun this notebook.")